# backtesting Scratch

Notebook for experimenting with the `backtesting` package

## Libraries

In [1]:
# stdlib

In [2]:
# 3rd party
import pandas as pd
import polars as pl
import ta

from backtesting import Strategy, Backtest
from backtesting.lib import resample_apply
from backtesting.lib import crossover

In [3]:
# project

## Strategy Setup

In [4]:
SMA = lambda x, n: ta.trend.sma_indicator(close=pd.Series(x), window=n)
RSI = lambda x, n: ta.momentum.rsi(close=pd.Series(x), window=n)

In [5]:
class System(Strategy):
    d_rsi = 30  # Daily RSI lookback periods
    w_rsi = 30  # Weekly
    level = 70
    
    def init(self):
        # Compute moving averages the strategy demands
        self.ma10 = self.I(SMA, self.data.Close, 10)
        self.ma20 = self.I(SMA, self.data.Close, 20)
        self.ma50 = self.I(SMA, self.data.Close, 50)
        self.ma100 = self.I(SMA, self.data.Close, 100)
        
        # Compute daily RSI(30)
        self.daily_rsi = self.I(RSI, self.data.Close, self.d_rsi)
        
        # To construct weekly RSI, we can use `resample_apply()`
        # helper function from the library
        self.weekly_rsi = resample_apply(
            'W-FRI',
            RSI,
            self.data.Close,
            self.w_rsi
        )
        
        
    def next(self):
        price = self.data.Close[-1]
        
        # If we don't already have a position, and
        # if all conditions are satisfied, enter long.
        if (not self.position and
            self.daily_rsi[-1] > self.level and
            self.weekly_rsi[-1] > self.level and
            self.weekly_rsi[-1] > self.daily_rsi[-1] and
            self.ma10[-1] > self.ma20[-1] > self.ma50[-1] > self.ma100[-1] and
            price > self.ma10[-1]):
            
            # Buy at market price on next open, but do
            # set 8% fixed stop loss.
            self.buy(sl=.95 * price)
        
        # If the price closes 2% or more below 10-day MA
        # close the position, if any.
        elif price < .98 * self.ma10[-1]:
            self.position.close()

## Data Import

In [6]:
(
    df := (
        pl
            .read_parquet(".data/binance/datasets/BTCUSDT_1m.parquet")
            .groupby_dynamic(
                index_column='Open time',
                every='1h',
                closed='left',
            )
            .agg(
               [
                   pl.col('Open').first(),
                   pl.col('High').max(),
                   pl.col('Low').min(),
                   pl.col('Close').last(),
                   pl.col('Volume').sum(),
               ]
            )
            .to_pandas()
            .set_index('Open time')
    )
).head()

,Open,High,Low,Close,Volume
Open time,,,,,
2017-08-17 04:00:00,4261.48,4313.62,4261.32,4308.83,47.181009
2017-08-17 05:00:00,4308.83,4328.69,4291.37,4315.32,23.234916
2017-08-17 06:00:00,4315.32,4345.45,4309.37,4324.35,7.229691
2017-08-17 07:00:00,4324.35,4349.99,4287.41,4349.99,4.443249
2017-08-17 08:00:00,4333.32,4377.85,4333.32,4360.69,0.972807


In [7]:
backtest = Backtest(df, System, commission=.002, cash=100_000)
backtest.run()
backtest.plot(filename='outputs/test.html', resample=False)

Row(id='1627', ...)

In [8]:
backtest.optimize(
    d_rsi=range(10, 35, 5),
    w_rsi=range(10, 35, 5),
    level=range(30, 80, 10),
    method='skopt'
)

Start                     2017-08-17 04:00:00
End                       2022-03-20 19:00:00
Duration                   1676 days 15:00:00
Exposure Time [%]                     4.64925
Equity Final [$]                    236816.32
Equity Peak [$]                  269516.15024
Return [%]                          136.81632
Buy & Hold Return [%]              857.786685
Return (Ann.) [%]                   20.639882
Volatility (Ann.) [%]               20.539122
Sharpe Ratio                         1.004906
Sortino Ratio                        2.487524
Calmar Ratio                         1.028504
Max. Drawdown [%]                  -20.067858
Avg. Drawdown [%]                   -2.455691
Max. Drawdown Duration      525 days 05:00:00
Avg. Drawdown Duration       25 days 05:00:00
# Trades                                   30
Win Rate [%]                        56.666667
Best Trade [%]                      21.603862
Worst Trade [%]                      -5.18964
Avg. Trade [%]                    

In [9]:
backtest.plot(filename='outputs/test.html', resample=False)

Row(id='3101', ...)